# **MOBILE ROBOTS Project Report**
**Group 38**

| **Name** | **SCIPER Number** | **Email** |
|---------|--------------------------|-----------|
| Killian Baillifard | 393835 | killian.baillifard@epfl.ch |
| Kael Murphy | 413744 | kael.murphy@epfl.ch |
| Moritz Tschudin | 327569 | moritz.tschudin@epfl.ch |
| Alex Gochely | 361135 | alex.gochely@epfl.ch |

### **Table of Contents**

1. [Introduction](#introduction)
2. [Environment](#environment)
3. [Vision](#vision)
4. [Global Navigation](#global-navigation)
5. [Motion Control](#motion-control)
6. [Local Navigation](#local-navigation)
7. [Filtering](#filtering)
8. [Visualisation Dashboard](#visualisation-dashboard)
9. [X](#XY)
10. [Conclusion](#conclusion)

## Introduction
The project for the course Basics of Mobile Robotics (BOMR) focuses on navigating the Thymio robot through an environment containing global obstacles, local obstacles, a defined start, and a goal. The main objective is to enable the robot to extract a global map from an overhead vision system, compute an (optimal) global path, and follow it autonomously. During navigation, the Thymio must also react to unexpected local obstacles placed in its path and handle special conditions such as kidnapping or losing access to the camera used for pose correction.

The project is structured into five modules, as defined in the official project description:

- **Vision** treated by *Kael*  
- **Global Navigation** treated by *Moritz*  
- **Motion Control** and **Local Navigation** treated by *Killian*  
- **Filtering** (Bayesian pose estimation) treated by *Alex*

## Environment

For our environment, we imagined the following scenario:  
Since last summer, extensive and noisy construction work has been taking place in the heart of the EPFL campus, on the Esplanade. These operations involve heavy vehicles such as excavators and trucks, as well as piles of rubble. We imagined a group of EPFL researchers aiming to deploy a mobile robot capable of autonomously delivering material across the construction site, an area where robotics and automation are still under-explored.

Because a construction site is a highly dynamic environment, the robot must be adaptable and capable of re-planning an optimal path every day, or even every hour, depending on changes in the environment. A drone would ideally provide an overhead view of the current state of the site, in our project, this drone is simulated by a camera mounted on the ceiling.

To simplify the scenario into a proof-of-concept using the Thymio robot, we abstract construction elements (trucks, excavators, rubble piles) into convex polygonal obstacles. Local obstacles, objects not visible to the camera or not included in the global map,  are represented by white cylinders topped with construction helmets, symbolizing construction workers who may unpredictably step into the robot’s path. These require the Thymio to perform local obstacle avoidance in real time. This setup is illustrated in figure below with (A) the real-world construction site, corresponding global obstacles such as vehicles and rubble in (B), and the resulting simplified polygons and map used for creating the environment in (C). *Source: See links on image.*

<p align="center">
  <img src="images/BOMR.png" width="90%">
</p>

The final environment therefore consists of:
- 4 ArUco markers defining the global frame  
- A white paper sheet including a zone(1250 × 740 mm) representing the workspace  
- 4 convex polygonal obstacles (global, static)  
- A start and goal, each represented by an ArUco marker  
- Local obstacles, represented by white cylinders with helmets  
This is illusrtated in figure below:
<p align="center">
  <img src="images/Setup_real_life.png" width="80%">
</p>


#### Use of AI Tools
Generative AI tools such as the BOMR AI Tutor, ChatGPT 5.1, and GitHub Copilot were used as supportive resources during the project. They were used for improving code structure, readability, commenting, debugging assistance, and refining the written text of the report. Their use was combined with the official course material of MICRO-452 Basics of Mobile Robotics and its Jupyter notebooks, which remained the primary reference for all technical decisions and implementations. Any additional external sources are cited directly in the report.

Next, we provide a detailed description of each of the five project modules.



## Vision

### 1. Overview  
The vision program is called from our demo file using a single function: `getVisionCoords(timeout, showDisplay)`. We use it to set up our world boundary and find the millimeter locations of our robot, goal, and obstacle vertices.

- **timeout**: number of frames to wait before timing out  
- **showDisplay**: boolean controlling whether the live camera display is shown  

#### 1.1. Vision acquisition pipeline  
The overall workflow used for vision acquisition is:

##### 1.1.1. Capture camera feed  
The program initializes a camera stream and continuously reads frames for processing.

##### 1.1.2. Detect ArUco markers  
We use **six ArUco markers** to represent system components:

- **IDs 0-3** → Operating zone corners  
- **ID 8** → Robot  
- **ID 9** → Goal  

##### 1.1.3. Build the operating zone  
The operating zone corners outline the **operating zone**, which is the boundary that the robot must stay within.

##### 1.1.4. Compute homography  
After detecting the world boundary, we apply a **homography transform** (`cv2.getPerspectiveTransform`) to flatten the camera image into a top-down plane and to convert our pixel coordinates into millimeters.

##### 1.1.5. Mask robot and goal for obstacle detection  
Before performing obstacle detection, the robot and goal markers are removed from the obstacle image:

- We locate the centers of ArUco IDs **8 (robot)** and **9 (goal)**.  
- A white circle is drawn over each marker (and the robot’s shadow) to stop them from being recognized as obstacles.  
- Obstacle detection is performed on this masked image.

##### 1.1.6. Find robot orientation
We need to find the angle of the robot when we query the vision program. Using the robot’s ArUco marker we form a vector in the robot’s forward direction and measure its angle relative to the bottom edge of the world boundary.

##### 1.1.7. Obstacle detection
We call the function `detectObstacles(obsFrame, zone, minArea)`, which takes in our mask and the minimum area threshold for a valid obstacle in pixels-squared. We blur, filter, and mask the frame to find the edges of the obstacles and draw their contours. If the contours enclose an area greater than our minimum, an obstacle instance is added to the obstacle list which is returned from this function.

##### 1.1.8. Drawing
Now that we have the pixel coordinates for our zone, robot, goal, and obstacles, we draw them using the draw helper functions. 

##### 1.1.9. Coordinate conversion
We first get all coordinates in the pixel space, then call `_extractFrameCoords(...)`, which uses `pixelToWorld` and `getWorldVerts(H)` to convert everything into millimeters before returning the final coordinate array. 

##### 1.1.10. Stability filtering  
We verify that our camera feed is stable. To avoid returning inconsistent or incorrect coordinates to the demo file, each frame's set of coordinates is added into a buffer. When the buffer contains `bufferLen = N` similar coordinates in a row, we consider the vision capture to be stable.

##### 1.1.11. Output  
After the stability check passes, the system returns four things:

1. A coordinate array in millimeters containing the corners, robot position, goal position, and obstacle vertices.  
2. The robot’s initial orientation angle in the world frame.  
3. The homography matrix used to convert pixels into millimeters.  
4. The pixel positions of the operating zone corners, used in the dashboard live feed.

#### 1.2. Fast robot pose acquisition  
Once the system has been initialized using `getVisionCoords(...)`, we assume the global obstacles and the corners of our operating zone will not change. Since we are expecting this behaviour, it doesn't make sense to redo this processing every time we capture a frame. Instead, we created a lightweight function `getRobotPositionMm()` that we call whenever we want an updated robot position from the camera.

##### 1.2.1. getRobotPositionMm()  
This helper function reads a frame from the cached camera, finds the robot marker, and computes its world coordinates and orientation using the $H$ transform from earlier. It returns the robot’s x-position, y-position, orientation angle, and a validity flag. If the robot is not visible, the function returns None for the position and angle, and False for `robotSeen`; if the robot is not in the frame the program will continue to run using only odometry.

#### 1.3. Shutdown and live frame access  

##### 1.3.1. stopVision()  
When the demo finishes, we can shut down the vision system using the `stopVision()` function. This stops the background camera thread, releases the video capture handle, and closes all windows.

##### 1.3.2. getLiveFrameBGR()  
This function allows other parts of the program to request the most recent camera frame at any time. We overlaid the theoretical path on top of the live feed so we can visually see how the robot is behaving.

### 2. Analysis

#### 2.1. Camera acquisition and threading

##### 2.1.1. OpenCV video pipeline

The camera is accessed through OpenCV’s `VideoCapture` using a fixed camera index. `VideoCapture.read()` pulls frames and converts them into a NumPy array, which will be accessed by our program.

When making the `CameraStream` object we use:

- Frame width and height: 640×480
- Framerate: 30 fps

##### 2.1.2. Threaded camera (CameraStream) design

Instead of calling `read()` in the vision processing loop, the program uses the `CameraStream` class that does the acquisition in its own thread. Essentially:

- The camera thread continuously pulls frames from the camera and caches the latest frame.
- The vision logic program reads from that cached frame whenever it needs.

We initialize the camera once and keep a single `CameraStream` running in the background continuously grabbing the latest frame. Different functions can then access this shared stream without opening or setting up the camera themselves. For example, we have slow, heavy processing in `getVisionCoords(...)`, while `getLiveFrameBGR(...)` only requires the raw frame with no additional processing. In our current main structure, these calls occur sequentially, and we wait for `getVisionCoords(...)` to return. However, the design supports a parallel architecture where a fast function can read new frames without being blocked by heavier functions. Later in the program, this becomes more important: functions like `getRobotPositionMm(...)` and `getLiveFrameBGR(...)` can grab the most recent frame from the camera, without having to reopen the camera or wait for another function to finish running. Even though we don’t really capitalise on this to parallelize more of our program, the cached-camera design is the most robust solution.

`CameraStream` class:

- The class has:
  - `self.cap`: the `VideoCapture` object tied to the camera.
  - `self.frame`: the newest BGR frame captured.
  - `self.lock`: a `threading.Lock` protecting access to `self.frame`.
  - `self.running`: a flag that controls the capture loop.
  - `self.thread`: the background thread object.

- The `_update` method:
  - Runs in a `while self.running` loop.
  - Calls `cap.read()` to grab the next frame.
  - Acquires lock and updates `self.frame`.
  - Small sleep timer to avoid using 100% of the CPU.

- The `read` method:
  - Acquires the same lock.
  - Returns a `.copy()` of the frame buffer.

What’s the point of the lock?:

- The main thread might try to read the frame at the same time as the camera thread is writing a new one.
- Without coordination, this could produce torn frames or corrupt data.

<p align="center">
    <img src="images/python_locking.png" alt="python_locking" width="400"/>
</p>

<p style="text-align: center;">
Mutex locking diagram (photo source: 
    <a href="https://www.geeksforgeeks.org/operating-systems/mutex-vs-semaphore/">https://www.geeksforgeeks.org/operating-systems/mutex-vs-semaphore/</a>
)
</p>

 As shown above, the lock makes sure that only one thread can access the shared `self.frame` array at a time. When a thread calls `lock.acquire()`, it either takes the lock or blocks until the other thread releases it. The main thread acquires the lock, copies `self.frame`, and releases it so the camera thread can keep capturing. The camera thread acquires the lock, writes the new frame, then releases it. As a result, the main thread never sees a partially written frame, and the camera thread never overwrites the array while it is being copied.

Even with the lock, once `read()` returns the main thread may spend some time processing the frame (for example, during ArUco detection). If we returned a reference to `self.frame` instead of `self.frame.copy()`, the camera thread might modify that array while the program is still processing it.

##### 2.1.3. Camera warm-up and exposure behaviour

If we immediately started running ArUco detection on the unstable frames generated right after the camera opens, the early detections would be inconsistent and delay the rest of our vision processing program. To avoid this, we discard the first chunk of frames captured after opening the camera. While discarding we initiate a small sleep per frame.

```python
warmup_frames = 30
for _ in range(warmup_frames):
  ok, _ = self.cap.read()
  if not ok:
    break
  time.sleep(0.02)
```

This does two things:

1. Gives the camera enough time to stabilize exposure, gain, and white balance.
2. Avoid over-using the CPU, similar to how we used the sleep timer in the `_update` method earlier.

After exiting the for-loop, the frames are closer to their steady state, which improves ArUco and obstacle detection later in the program.

#### 2.2. ArUco marker detection

##### 2.2.1. ArUco dictionaries and encoding

An ArUco marker is a QR-code-esque identifier typically used for computer vision applications:

- A black border defines the region.
- The interior is a binary grid encoding an ID.
- It has redundancy and error correction:
  - If some cells are partially blocked or corrupted by noise, the decoder can still recover the ID.
  - The orientation of the marker can be deduced since the binary pattern only matches one orientation.

<p align="center">
    <img src="images/Aruco_example.png" alt="Aruco" width="200"/>
</p>

<p style="text-align: center;">
Example ArUco marker (photo source: 
    <a href="https://chev.me/arucogen/">chev.me/arucogen</a>
)
</p>


The dictionary that we use ("DICT_4X4_50") contains:

- The size of the inner grid (4x4).
- The number of valid patterns and how they map to IDs, in our case 0-49.
- Each marker pattern is different enough from the others that the camera doesn't confuse them.

In our system:

- IDs 0-3 are the operating zone corners [bottom-left = 3, bottom-right = 2, top-right = 1, top-left = 0].
- ID 8 is the robot.
- ID 9 is the goal.

##### 2.2.2. Detection steps (`detectAruco`)

The `detectAruco` function implements the ArUco detection pipeline, this is all internal to OpenCV, so we didn't write the code for this. However, the method OpenCV uses is explained below:

1. **Conversion to greyscale**

ArUco markers only store information in black & white intensity as seen above, so OpenCV turns the three BGR channels into a single greyscale channel:

$$
I(x,y) = 0.114\,B(x,y) + 0.587\,G(x,y) + 0.299\,R(x,y)
$$

Where:

- $I(x,y)$: greyscale intensity at pixel (x,y)
- $B(x,y),G(x,y),R(x,y)$: blue, green, and red channel intensities at pixel (x,y)
- $0.114, 0.587, 0.299$: channel coefficients used by OpenCV

I found the coefficients in the OpenCV documentation at:  
https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html

**Why convert to greyscale:**  
- ArUco markers only encode information through black & white intensity, not colour. Using greyscale makes the filtering process easier since the algorithm only separates “dark” from “light”,  rather than processing three colour channels.
- Greyscale removes colour-specific noise and produces cleaner edges for thresholding and contour detection.

2. **Detector initialization**

OpenCV loads the `DICT_4X4_50` dictionary.  
This dictionary restricts detection to exactly 50 known 4×4 bit patterns (IDs 0-49)

3. **Looking for ArUco markers**

a. **Adaptive thresholding**

Conceptually, ArUco detection applies adaptive thresholding (local mean thresholding) to handle uneven lighting:

$$
T(x,y) = \frac{1}{|N|} \sum_{(u,v)\in\N} I(u,v) - C
$$

Where:

- $T(x,y)$: threshold at pixel (x,y)
- $N$: local neighbourhood window
- $|N|$: number of pixels in the neighbourhood
- $I(u,v)$: greyscale intensity at neighbour pixel (u,v)
- $C$: constant offset to bias the threshold

b. **Contour filtering**

After thresholding, the image is binarized with pixels either being black (0) or white (255). OpenCV finds areas of the image containing boundaries between white and black regions, then it "walks" along these edges and returns its path (the contour). Most contours in the image are not ArUco markers, so OpenCV filters contours based on:

- Area  
- Convexity  
- Approximate shape

c. **Warping and bit sampling**

Initially the contour is tilted / skewed, each candidate marker is rectified to a square via a homography, so that the inner code grid can be sampled on a regular 4×4 grid. The H-transform turns each contour into a perfect square via:

$$
\begin{bmatrix}
x' \\[4pt]
y' \\[4pt]
w'
\end{bmatrix}
=
H
\begin{bmatrix}
x \\[4pt]
y \\[4pt]
1
\end{bmatrix},
\qquad
H =
\begin{bmatrix}
h_{00} & h_{01} & h_{02} \\
h_{10} & h_{11} & h_{12} \\
h_{20} & h_{21} & h_{22}
\end{bmatrix}
$$

$$
x_{\text{norm}} = \frac{x'}{w'}, 
\qquad
y_{\text{norm}} = \frac{y'}{w'}
$$

$$
x_{\text{norm}}
=
\frac{
h_{00}x + h_{01}y + h_{02}
}{
h_{20}x + h_{21}y + h_{22}
},
\qquad
y_{\text{norm}}
=
\frac{
h_{10}x + h_{11}y + h_{12}
}{
h_{20}x + h_{21}y + h_{22}
}.
$$

Where the homography parameters are defined as:

- $h_{00}, h_{01}$: control horizontal scaling, rotation, and skew  
- $h_{10}, h_{11}$: control vertical scaling, rotation, and skew  
- $h_{02}, h_{12}$: control translation (shifting the warped image)  
- $h_{20}, h_{21}$: encode perspective tilt (how much lines "converge")  
- $h_{22}$: normalization term (usually set to 1)

And $w'$ is a scale factor used to turn the coordinates back into pixel coordinates after the transform.

Homography computation:

Given 4 corner points in the image $(x_i, y_i)$ and the 4 corner points of a perfect square $(X_i, Y_i)$, the transform satisfies:

$$
\begin{bmatrix}
X_i \\ Y_i \\ 1
\end{bmatrix}
\sim
H
\begin{bmatrix}
x_i \\ y_i \\ 1
\end{bmatrix}.
$$

In general, a homography with four point correspondences is solved using a Direct Linear Transform (DLT), and intensities at non-integer locations are obtained by bilinear interpolation. Conceptually, this is what happens when OpenCV rectifies the marker and samples the inner grid:

$$
I(x,y) = 
w_{00} I(\lfloor x \rfloor,\lfloor y \rfloor)
+ w_{01} I(\lfloor x \rfloor,\lfloor y \rfloor+1)
+ w_{10} I(\lfloor x \rfloor+1,\lfloor y \rfloor)
+ w_{11} I(\lfloor x \rfloor+1,\lfloor y \rfloor+1)
$$

Where:

- $I(x,y)$: intensity at fractional coordinate (x,y)
- $\lfloor x \rfloor,\lfloor y \rfloor$: nearest integer pixel locations
- $w_{ij}$: interpolation weights (sum to 1)

The homography transform paired with the bilinear interpolation returns the intensity value of the sub-pixel brightness. This will be used to determine whether this sub-pixel is a part of the ArUco code.

d. **Dictionary matching**

The sampled 4×4 bits form a candidate pattern $M$. OpenCV compares this against each dictionary entry $D_k$ to ensure that only one ID is matched to what we have identified:

$$
d_H(M, D_k) = \text{number of bit positions where } M \neq D_k
$$

Where:

- $M$: extracted 4×4 bit matrix
- $D_k$: k-th dictionary pattern
- $d_H$: Hamming distance (bit mismatches)

4. **Output structure**

The detector returns `ids`, a list of IDs that were found in the frame, and `corners`, a list containing the corners for each of the ArUco markers in pixel-coordinates.

We compute the marker center as the mean of the four corners:

$$
c = \frac{1}{4} \sum_{k=0}^{3} p_k
$$

Where:

- $p_k$: k-th corner point (2D pixel coordinate)
- $c$: computed marker center

#### 2.3. Operating zone and coordinates

##### 2.3.1. Zone definition from markers 0-3

The operating zone for our robot is defined by the four corner ArUco markers:

- IDs 0, 1, 2, 3 sit at the corners of the physical board.
- The code constructs a `zone` dictionary where:
  - `zone["isComplete"]` is true when all corners are detected.
  - `zone["missing"]` is a list of the missing corner IDs.
  - `zone["corners"]` is an ordered list of the pixel centers of the corner markers.

The order is forced even if the IDs are returned in a different sequence because:

1. Homography transform assumes that the nth pixel corner corresponds to the nth world corner. If the order is wrong, the homography will warp the board incorrectly.
2. When we draw the polygon representing the zone, we want a simple counter-clockwise loop that does not self-intersect; consistent ordering guarantees that.
3. When we return the list to `demo.py`, the global navigation program expects all polygons to be counter-clockwise.

##### 2.3.2. World coordinate system

The world coordinate system is in millimeters, and the size of it is set by the physical dimensions we chose to use on our background sheet of paper. The known width and height of our operating zone are set as constants:

- `widthMm` ~ 1255 mm
- `heightMm` ~ 740 mm

The coordinates of the four board corners in world space are:
  - (0, 0) = bottom-left
  - (widthMm, 0) = bottom-right
  - (widthMm, heightMm) = top-right
  - (0, heightMm) = top-left

All points inside the operating zone can then be expressed in millimeters after finding the suitable homographic transform.

#### 2.4. Homography and pixel to world coordinate mapping

##### 2.4.1. Homography transform

As discussed earlier in Section 2.2.2 ("Looking for ArUco markers"), the homography transformation accounts for when the camera is not perfectly orthogonal to the operating zone, if there is any rotation of the board, and scaling differences that happen as you move out of the center of the camera's vision. For our project, the problem was that we needed our operating zone to have a uniform scale so that our Thymio could be sent waypoints in millimeters anywhere in the zone; the $H$ transform solves this:

<p align="center">
    <img src="images/homography_easy.png" alt="Homography" width="400"/>
</p>

<p style="text-align: center;">
Homography example (photo source: 
    <a href="https://mattmaulion.medium.com/homography-transform-image-processing-eddbcb8e4ff7">https://mattmaulion.medium.com/homography-transform-image-processing-eddbcb8e4ff7</a>
)
</p>

Similarly to the ArUco codes:
- The zone corner pixel centers (`zone["corners"]`) are the source points.
- The world corner coordinates are the destination points.
- `cv2.getPerspectiveTransform` takes these and solves for $H$.

To map a pixel point pt = (x, y) into millimeter world coordinates we form a $1\times 1\times 2$ array to pass into `cv2.perspectiveTransform`, OpenCV multiplies $H$ by $[x, y, 1]^T$ internally and returns the result, finally it normalizes by the third coordinate $w'$ to yield $(X, Y)$. It returns (X_mm, Y_mm) which is the position in millimeters. We also use the inverse of the $H$ transform to convert our path (in mm) back into pixel coordinates for our dashboard.

#### 2.5. Robot pose estimation

##### 2.5.1. Robot position and orientation

The robot has an ArUco marker with ID 8 and using `centerMap[8]` we get the pixel center. To convert to world position we use `pixelToWorld(center_pixel, H)` and the robot’s estimated position (x_mm, y_mm) is returned. We find the orientation of the robot using the center of the top edge as the forward direction. Using the ArUco corner ordering convention we:

1. Take the corners for ID 8 from `cornerMap`, index 0 is top-left and index 1 is top-right.
2. Compute the midpoint of these two corners in pixel coordinates.
3. Convert:
   - The marker center to world coordinates: (cx, cy).
   - The midpoint of the top edge to world coordinates: (tx, ty).
4. Create a vector from robot center to the edge midpoint:
   - dx = tx - cx
   - dy = ty - cy
5. Compute the orientation angle:
   - $\theta$ = atan2(dy, dx)

$\theta$ is defined in the world coordinates by:

- $\theta$ = 0 when the robot faces along +X (parallel to zone bottom edge facing the right).
- $\theta$ increases counter-clockwise:
  - $\pi/2$: robot faces +Y (towards the top of the board).
  - $\pi$: robot faces -X (leftwards).
  - $3\pi$/2: robot faces -Y (downwards).

The ArUco marker is attached to the robot so that its center is roughly in the same position as the robot’s center of rotation and its top edge points in the robot’s forward direction. If the marker is misaligned then $\theta$ will have a constant offset with the real forward heading of the robot.

#### 2.6. Obstacle detection and processing

##### 2.6.1. Colour-space and masking choices

Obstacle detection is performed directly in BGR space with a Gaussian blur that reduces noise and `cv2.inRange` is used with a lower and upper threshold on the B, G, and R channels. We chose to stay in BGR rather than convert to HSV for simplicity. The obstacle colour was distinct enough that direct BGR thresholds were easy to tune by trial and error. Additionally, the system uses a binary zone mask so that objects or noise outside the world boundary do not show up as obstacles. 

Before obstacle detection, the robot and goal markers are painted over with white circles on the obstacle frame:

<p align="center">
    <img src="images/obstacle_photo.png" alt="Obstacle" width="300"/>
</p>

We used the paint-over to stop the ArUco markers and the robot's shadow from being detected as obstacles. The radius was chosen based on the size of the box created by the marker’s corners; it was scaled to cover the marker and nearby shadows.

##### 2.6.2. Blurring

A Gaussian blur is applied to the BGR image before applying the colour threshold. Gaussian blur replaces each pixel with a weighted average of its neighbours. It smooths out small changes in intensity and colour due to sensor noise or texture. Finally, it limits the amount of isolated pixels that may be classified as an obstacle. The kernel size (5x5) was chosen because it is large enough to smooth out noise at the pixel scale, but it is small enough to keep the edges of obstacles.

<p align="center">
    <img src="images/gaussian_blur.png" alt="Gaussian" width="400"/>
</p>

<p style="text-align: center;">
Gaussian blur example (photo source: 
    <a href="https://hackaday.com/2021/07/21/what-exactly-is-a-gaussian-blur/">https://hackaday.com/2021/07/21/what-exactly-is-a-gaussian-blur/</a>
)
</p>

$$
I_{\text{blur}}(x,y)
=
\sum_{i=-2}^{2}
\sum_{j=-2}^{2}
G_{\text{5×5}}(i,j)\, I(x+i,\, y+j)
\qquad
G_{\text{5×5}} =
\frac{1}{273}
\begin{bmatrix}
1 & 4 & 7 & 4 & 1 \\
4 & 16 & 26 & 16 & 4 \\
7 & 26 & 41 & 26 & 7 \\
4 & 16 & 26 & 16 & 4 \\
1 & 4 & 7 & 4 & 1
\end{bmatrix}
$$

We apply a $5\times 5$ Gaussian blur by taking each pixel and combining it with the surrounding twenty-four neighbours using the fixed weights supplied by OpenCV. The weight matrix approximates a Gaussian distribution. For each pixel, we multiply every neighbour by the corresponding weight and add the results. The weights are such that they sum to one, which keeps the image brightness the same.

##### 2.6.3. Colour thresholding

After blurring, obstacle detection is performed by applying `cv2.inRange` directly in the BGR colour space. This function compares each pixel against a lower and upper BGR bound chosen earlier:

`mask = cv2.inRange(blurred, lowerBGR, upperBGR)`

Pixels with Blue, Green, and Red channel values all within the ranges are set as 255, and everything else is set as 0. This is similar to the binarization step used in ArUco detection, we reduce the image to a binary mask so later steps (morphology and contours) operate on clean regions. It is then passed into our morphological functions, where the mask will be cleaned up so our contour detection returns the obstacles we are interested in. 

##### 2.6.4. Morphological operations and findContours

After thresholding we clean up the mask using:

```python
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
```
The elliptical kernel defines the neighbourhood used for smoothing. An ellipse was chosen because it produces natural-looking shapes that match our obstacles better than a square kernel. The first operation is closing, which fills small holes and connects small gaps so that each obstacle turns into a solid region. Closing is applied twice to ensure that obstacles broken by thresholding are repaired. The second operation is opening, which removes small isolated noise pixels and smooths the edges of the cleaned regions.

We then extract the outline of each obstacle using `findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)`:

- Retrieval mode: external contour only which returns the outer boundary of each obstacle and ignores any holes.
- Approximation method: chain approximation reduces the number of stored boundary points and keeps only the essential outline.

Each contour represents an obstacle. The system then computes the area of each contour and filters them using two thresholds:

- A minimum area (minArea).
- A maximum area (maxArea).

Small areas usually correspond to noise or artifacts, and large areas mean that there are likely lighting problems and that large sections of our zone are being detected as obstacles. Contours within the acceptable range are turned into `Obstacle` objects and added to the list.

##### 2.6.6. Polygon approximation

Each `Obstacle` object simplifies its contour into a smaller polygon using the `approxPolyDP(self.contour, epsilon, True)`:

```python
if self._verts is None:
  peri = cv2.arcLength(self.contour, True)
  epsilon = 0.02 * peri
  approx = cv2.approxPolyDP(self.contour, epsilon, True)

  pts = [(float(p[0][0]), float(p[0][1])) for p in approx]
```

Where:
- peri is the perimeter length of the contour.
- epsilon is the maximum deviation allowed for the approximation.
- approx is the new simplified polygon.
- pts is the list of (x,y) in pixel coordinates.

Benefits of using a simplified polygon approximation are:

- Eliminates unnecessary detail in the contour, small variations on the boundary are removed.
- Produces obstacle polygons with a reasonable number of vertices.
- Gives stable representation of obstacle vertices for repeated trials.

After approximation, the obstacle vertices are “snapped” to a pixel grid. Each corner is rounded to the nearest multiple of a small grid spacing (we used 3), and vertices that end up close to each other are merged to further simplify the obstacle:

```python
grid = 3
snapped = [
    (int(round(x / grid) * grid), int(round(y / grid) * grid))
    for (x, y) in pts
]

merged = []
mergeDist = 3
for x, y in snapped:
    if not merged:
        merged.append([x, y])
        continue

    found = False
    for v in merged:
        dx = x - v[0]
        dy = y - v[1]
        if dx * dx + dy * dy <= mergeDist * mergeDist:
            v[0] = int(round((v[0] + x) / 2))
            v[1] = int(round((v[1] + y) / 2))
            found = True
            break

    if not found:
        merged.append([x, y])

self._verts = [(vx, vy) for vx, vy in merged]
```

##### 2.6.7. Obstacle class

Once the polygon vertices are defined in pixel coordinates, we convert them to world coordinates using the same homography $H$ used for markers. `pixelToWorld` is called and the resulting world coordinates are added to a list. The vertices are then sorted by:

1. Computing the centroid.
2. For each vertex, compute the angle from centroid to vertex.
3. Sort by this angle to obtain counter-clockwise order of vertices.
4. Find the bottom-left vertex (lowest Y, then lowest X), and rotate the list so that this vertex is first.

Why this ordering?

1. Consistent vertex labeling regardless of detection order.
2. We use Shapely to draw the polygons for path planning, we follow the convention of defining polygons with counter-clockwise vertices to avoid ambiguity and to match the expectations of our path-planning code. This is expanded on further in the global navigation section.

The final representation of an obstacle in the coordinate array is as a set of rows:

- type: "vertex"
- polygon ID: "polyX"
- vertex label: "1", "2", ...
- x_mm, y_mm: world coordinates in millimeters

#### 2.7. Stability filtering and robustness

##### 2.7.1. Motivation for temporal filtering

The ArUco detector occasionally misses a marker if it is partially blocked or blurry and obstacle extraction can be sensitive to lighting changes. If we returned vision coordinates using one frame:

- Corners may disappear and reappear.
- Obstacles could jump around.
- Inconsistent robot and goal positions and orientations.

To avoid this, we used a buffer that stores frames and compares the coordinate values to see if the camera feed is stable. Until then, the system collects frames and updates the buffer.

```python
def obstaclesAreSimilar(obsArr1, obsArr2, tol=50.0):
  if len(obsArr1) != len(obsArr2):
    return False
  for r1, r2 in zip(obsArr1, obsArr2):
    if r1[2] != r2[2] or r1[1] != r2[1]:
      return False
    x1, y1 = round(r1[3], 1), round(r1[4], 1)
    x2, y2 = round(r2[3], 1), round(r2[4], 1)
    if abs(x1 - x2) > tol or abs(y1 - y2) > tol:
      return False
  return True
```

##### 2.7.2. Buffer design and stability

The system maintains a buffer `coordBuf` of the last N coordinate arrays. Each element is the array with the coordinates of the corners, robot, goal, and obstacle vertices.

When a new coordinate array `coords` is available, the code compares the obstacle coordinates to those held in the last buffer entry. We extract array rows where type == "vertex" and check that the same number of obstacle vertices are present, IDs and labels match, and the positions of corresponding vertices are within our tolerance of 50 mm. We chose 50 mm as a tolerance because it is large enough to ignore minor detection noise, but small enough that obstacles can't “jump” to a different location and still be considered stable. If the new obstacles are within our tolerance they are appended to the buffer. If they aren't within our tolerance, the buffer is reset to contain just the new frame. Once the buffer has N similar elements, the system checks that all entries are similar to the first one:

- If yes, our feed is stable and we can return these coordinates to the demo file. 
- If no, try again.

We are able to toggle latency and stability by modifying our hardcoded `bufferLen` values:

- Increase bufferLen:
  - Increase stability, function takes longer to get stable coordinates.
- Decrease bufferLen:
  - Might have errors in our detection, but will return faster.

We adjusted `bufferLen` throughout our testing, when obstacle detection was noisy we increased it to ensure stability. When detection was reliable we decreased it to speed up the program. 

#### 2.8. Fast robot pose acquisition

Once the vision system has been initialized and the homography has been computed, the heavy processing does not need to be repeated every time we want to update the robot’s position. The function `getRobotPositionMm()` provides an easy way to fetch the robot’s pose using the cached camera and homography transformation. The function returns x_mm, y_mm, theta, robotSeen; robotSeen is a boolean flag that is used to determine whether the camera can be used for position determination.

##### 2.8.1. Requirements and frame acquisition loop

This function needs:

- `VISION_CAMERA`: the background `CameraStream` object (Section 2.1).
- `VISION_H`: the pixel to world homography matrix computed during initialization (Section 2.4).

The function retrieves the newest frame from the camera:

```python
frame = VISION_CAMERA.read()
if frame is None:
    time.sleep(0.01)
    continue
```

This matches the locking behaviour from earlier. If no frame is ready, wait; otherwise process it.

##### 2.8.2. Detecting the robot marker

Aruco detection is applied to the frame:

```python
_, centerMap, cornerMap, _ = detectAruco(frame)
```

If the robot marker (ID 8) isn't detected:

```python
return None, None, None, False
```

##### 2.8.3. Coordinate conversion and robot orientation

The robot’s pixel center is mapped into world coordinates using the cached homography:

```python
robotWorld = pixelToWorld(centerMap[ROBOT_ID], VISION_H)
x_mm, y_mm = robotWorld

```

The orientation procedure is the same as described earlier:

1. Compute the midpoint of the top edge of the robot marker.
2. Convert the midpoint into world coordinates.
3. Make the vector from robot center to the midpoint.
4. Compute the heading using `atan2`.

```python
topMidPx = (rCorners[0] + rCorners[1]) / 2.0
topMidWorld = pixelToWorld(topMidPx, VISION_H)
dx = topMidWorld[0] - x_mm
dy = topMidWorld[1] - y_mm
theta = math.atan2(dy, dx)
```

#### 2.9. Live frame acquisition

##### 2.9.1. getLiveFrameBGR() function

A function to get the latest frame from the cached camera. Used alongside the inverse $H$ transform to turn the path into pixel coordinates from world coordinates.

```python
def getLiveFrameBGR():
    """return the latest BGR frame from the cached CameraStream."""
    global VISION_CAMERA
    if VISION_CAMERA is None:
        raise RuntimeError(
            "init world error"
        )
    return VISION_CAMERA.read()
```

## Global Navigation

In this section, we describe the design and implementation of the global navigation module. Its purpose is to compute a collision-free geometrically optimal path for the Thymio robot, based on the environment reconstructed by the Vision Module.

Our approach uses a Visibility Graph combined with A* search, as introduced in the course. Obstacles detected in the Esplanade construction scenario (e.g., excavators, transport vehicles, rubble piles) are abstracted as convex polygons, allowing efficient geometric reasoning.

We use the [Shapely library](https://shapely.readthedocs.io/en/stable/)
for computational geometry (e.g., `Polygon`, `LineString`, intersection tests), which greatly simplifies handling polygonal obstacles.

### Chosen Planning Approach: Visibility Graph
##### Advantages
- Computes the shortest geometric path in the free space, rather than on a discretized grid.
- Requires only a few nodes given a simplified environment.
- Generates a compact list of waypoints, suitable for robot execution.
- Works well with convex polygonal obstacles.

##### Limitations
- Requires precise polygon vertices from the Vision Module.
- Robot modeled as a point requires obstacle inflation to avoid collisions.
- Path must be recomputed if global obstacles would be moved.

### Overview of the Planning Pipeline
The global planner consists of the following stages:

1. Input from Vision Module
2. Polygon orientation normalization (CCW)
3. Construct obstacle polygons
4. Inflate polygons by a factor ε
5. Extract nodes (vertices + start + goal)
6. Build the Visibility Graph
7. Run A* search
8. Return final waypoint list

### 1. Input from the Vision Module
The Vision Module provides obstacle vertices as well as start and goal coordinates in the format:

`type, id, label, x, y`

Example:

```
vertex, poly1, A, 123.4, 567.8
vertex, poly1, B, 200.0, 540.0
start, start, S,  50.0, 900.0
goal,  goal,  G, 900.0,  60.0
```

These coordinates are already converted into the world coordinate system in millimeters using camera calibration.

### 2. Polygon Orientation (CCW)
Shapely requires polygons to have consistent vertex ordering.
We ensure counter-clockwise (CCW) orientation using the signed area (shoelace) method:

\begin{array}{l}
A = \frac{1}{2} \sum_{i=1}^{n} \left( x_i y_{i+1} - x_{i+1} y_i \right)
\end{array}

with the convention:
\begin{array}{l}
(x_{n+1},\, y_{n+1}) = (x_1,\, y_1)
\end{array}


- If \( A > 0 \) → polygon is CCW  
- If \( A < 0 \) → reverse vertex order  

All polygons are therefore normalized to CCW orientation.

Short helpers: CCW + small inflated polygon snippet
```python
# Short helpers: shoelace area and enforce counter-clockwise
def signed_area(ring):
    a = 0.0
    for (x1,y1), (x2,y2) in zip(ring, ring[1:]+ring[:1]):
        a += x1*y2 - x2*y1
    return 0.5*a

def ensure_ccw(ring):
    return ring if signed_area(ring) > 0 else list(reversed(ring))
```

Source: [The Shoelace Algorithm](https://www.101computing.net/the-shoelace-algorithm/).

### 3. Constructing Polygons
We reconstruct each obstacle as a Shapely `Polygon`.

This allows:

- Robust collision detection
- Buffering / inflation
- Geometric operations
- Clean visualization

We also render the reconstructed map for debugging and verification. See figure below:

<p align="center">
  <img src="images/polygons_0.png" width="50%">
</p>

### 4. Obstacle Inflation
Because the global planner models the Thymio as a point-mass robot (as in the course), each obstacle must be inflated by at least half of the robot’s footprint to ensure collision-free motion. In practice, the inflation radius ε was tuned empirically, and the details of this parameter selection are discussed later in the section Parameters, tuning and how we chose ε (epsilon).

Equivalent to Minkowski sum:

\begin{array}{l}
\tilde{\mathcal{O}} = \mathcal{O} \oplus B(\varepsilon)
\end{array}

is Shapely's buffer operation:

```
poly.buffer(epsilon, join_style=2, resolution=1)
```

This line performs the Minkowski sum inflation while preserving sharp corners (join_style=2) and minimizing unnecessary vertices (resolution=1), which keeps the configuration-space polygons clean and computationally efficient for visibility graph construction.
Note that poly0 is not inflated as it stays the outer boundary, the world boundary.

See figure below for inflated polygons choosing different values for ε:

<p align="center">
  <img src="images/polygons_4.png" width="100%">
</p>

Minimal inflate_polygons demonstration:

```python
from shapely.geometry import Polygon

def inflate_ring(ring, eps_mm=20.0, join_style=2, resolution=1):
    """Return buffered exterior ring coordinates (preserve sharp corners)."""
    poly = Polygon(ring)
    buf = poly.buffer(eps_mm, join_style=join_style, resolution=resolution)
    if buf.geom_type == "MultiPolygon":
        # choose the largest part (rare)
        buf = max(buf.geoms, key=lambda p: p.area)
    coords = list(buf.exterior.coords)[:-1]  # drop closing vertex
    return ensure_ccw(list(coords))
```

Source: [Wikipedia](https://en.wikipedia.org/wiki/Minkowski_addition)

Source: [ME-425 Robust MPC 2](https://moodle.epfl.ch/pluginfile.php/1521570/mod_resource/content/1/i%20Robust%20MPC%202.pdf)

Source: [Shapely's buffer fct](https://shapely.readthedocs.io/en/stable/reference/shapely.buffer.html)

### 5. Node Extraction
We define nodes as:
- All vertices of inflated obstacles except the outer boundary poly0
- The start coordinate
- The goal coordinate

These are stored as:
```
node_coords = [(x0, y0), ...]
node_labels = ["poly1_0", ..., "start", "goal"]
```

The outer boundary poly0 is only used as a world constraint in the visibility check and is not turned into graph nodes.

### 6. Building the Visibility Graph
For every pair of nodes `(i,j)`, we test whether the segment connecting them is obstacle-free.
We use from Shapely:
- `LineString(segment).crosses(polygon)`
- `LineString(segment).within(polygon)`

We allow segment-obstacle tangency (parallel to edges) since obstacles are already inflated.

Helper functions are :
- `segment_visible()`
- `euclidean_distance(i, j)`
- `build_neighbors(...)`

The adjacency list defines the Visibility Graph.

Example neighbor count:
```
Node  0 (poly1_0 ) has 5 neighbors
Node 19 (start   ) has 7 neighbors
Node 20 (goal    ) has 4 neighbors
```

A visualization confirms correct visibility edge construction. See figure below:

<p align="center">
  <img src="images/polygons_1.png" width="50%">
</p>

Short segment visibility & neighbor builder:

```python
from shapely.geometry import LineString

def segment_visible(p, q, world_poly, obstacle_polygons):
    seg = LineString([p,q])
    # Must remain inside the world
    if not (seg.within(world_poly) or seg.touches(world_poly)):
        return False
    # Disallow segments that cross obstacle interiors
    for obs in obstacle_polygons:
        if seg.crosses(obs) or seg.within(obs) or obs.contains(seg):
            return False
    return True

def build_neighbors(node_coords, world_poly, obstacles):
    N = len(node_coords)
    neighbors = [[] for _ in range(N)]
    for i in range(N):
        p = node_coords[i]
        for j in range(i + 1, N):
            q = node_coords[j]

            if segment_visible(p, q, world_poly, obstacle_polys):
                d = euclidean_distance(p, q)
                neighbors[i].append((j, d))
                neighbors[j].append((i, d))

    return neighbors
```

### 7. A* Path Planning
We apply A* search on the visibility graph.
The cost function is:

\begin{array}{ll}
f(n)=g(n)+h(n)
\end{array}

Where:

- $g(n)$: accumulated distance from start  
- $h(n) = \| n - \text{goal} \|_2$: Euclidean distance heuristic

The heuristic uses Euclidean distance (straight-line), which never overestimates the true shortest distance on a graph whose edge costs are Euclidean distances.

Although turn penalties could be included, the visibility graph already contains few nodes and produces nearly straight-line paths, making additional penalties less necessary.

Minimal A* snippet (The actual implementation adds logging and counts expansions, but is structurally identical to this minimal version.):
```python
import heapq, math

def heuristic(node_idx, goal_idx, node_coords):
    """Euclidean distance from node to goal."""
    x1, y1 = node_coords[node_idx]
    x2, y2 = node_coords[goal_idx]
    return math.hypot(x1 - x2, y1 - y2)

def astar(neighbors, node_coords, start_idx, goal_idx):
    """A* search on a visibility graph."""
    g_cost = {start_idx: 0.0}        # best known cost from start
    came_from = {}                   # child -> parent
    explored = set()                 # closed set

    # priority queue of (f_cost, g_cost, node_idx)
    open_set = []
    h0 = heuristic(start_idx, goal_idx, node_coords)
    heapq.heappush(open_set, (h0, 0.0, start_idx))

    while open_set:
        f_curr, g_curr, current = heapq.heappop(open_set)

        if current in explored:
            continue
        explored.add(current)

        if current == goal_idx:
            break

        for neighbor, weight in neighbors[current]:
            tentative_g = g_curr + weight
            if neighbor not in g_cost or tentative_g < g_cost[neighbor]:
                g_cost[neighbor] = tentative_g
                came_from[neighbor] = current
                h = heuristic(neighbor, goal_idx, node_coords)
                f = tentative_g + h
                heapq.heappush(open_set, (f, tentative_g, neighbor))

    if goal_idx not in came_from and start_idx != goal_idx:
        return None

    # reconstruct path
    path = [goal_idx]
    current = goal_idx
    while current != start_idx:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path

```

### 8. Output and Integration with the Control Module
Running A* on the visibility graph yields a waypoint sequence such as:
```
[(10.0, 900.0), (75.3, 590.1), (595.6, 176.6), (900.0, 10.0)]
```
This list of $(x,y)$ waypoints is returned by `compute_global_path(map_array, epsilon_mm)` as a NumPy array and passed to the Motion Control module as the global plan to be tracked by the Thymio.

Below is an example showing the optimal A* path (red) in the inflated polygonal environment:

<p align="center">
  <img src="images/polygons_2.png" width="50%">
</p>

### Implementation Summary & File Structure
The global navigation module is implemented across two files:

#### `globalnav.py`: core planning pipeline
This file implements the complete global planning pipeline:
  - Input parsing
  - CCW normalization
  - Polygon construction (Shapely)
  - Obstacle inflation using a robot safety margin
  - Node extraction
  - Visibility graph construction
  - A* path search
It exposes two functions:
 - `compute_global_path(map_array, epsilon_mm)` Runs the full pipeline and returns only the final waypoint list (used by the Control module).
 - `compute_global_path_with_debug(map_array, epsilon_mm)` Same computation, but also returns all intermediate data needed for plotting (raw/inflated polygons, nodes, visibility edges, etc.).

#### `globalnav_plot.py`: visualization helper
Responsible for plotting the global environment and the A* result.

- `setup_globalnav_plot(map_array, epsilon_mm, ...)`:
   - Calls `compute_global_path_with_debug(...)` exactly once during GUI initialization,
   - Extracts from debug data to draw:
     - Original vs inflated polygons
     - Optional visibility edges
     - Start/goal
     - The A* optimal path

- Creates Matplotlib objects (lines/arrows) that the GUI updates live for:
   - Odometry trajectory
   - Camera trajectory
   - Robot pose arrows (odometry and camera)

### Single A* Call During GUI Initialization
The dashboard calls `setup_globalnav_plot(...)` a single time.
Inside this function:

- `compute_global_path_with_debug()` is executed once
- The resulting path and debug data are cached
- The control module reuses this exact path during execution

Why this matters
- No duplicate A* calls → avoiding nconsistent paths  
- The same inflation radius ε is shared by the:
  - Planner
  - Visualization
  - Motion controller
- Ensures agreement between the displayed map and the robot’s executed trajectory
- Prevents subtle bugs such as mismatched inflation radii or inconsistent path computation.

### Parameters, Tuning, and Choice of ε
Several parameters influence the reliability and safety of the global planner:

- Inflation radius ε (mm)

  Expands obstacles to account for the robot’s footprint and keep paths at a safe distance.

   - Small ε → paths may pass too close to obstacles
   - Large ε → obstacles can merge and block corridors

  Values between 20-90 mm were tested, and ε was tuned empirically based on the robot’s actual clearance and behaviour.

- Buffer settings (join_style=2, resolution=1)

Chosen to keep sharp polygon corners (mitre join) and avoid generating too many vertices.
This gives fast, stable inflation that approximates the Minkowski sum without creating overly round obstacles.

- Visibility graph construction

Nodes are taken from the inflated polygons, ensuring that straight-line edges remain safely outside real obstacles.
Segments touching obstacles are allowed and segments crossing or lying inside obstacles are rejected.

- A* heuristic

The Euclidean distance is admissible and consistent, guarantees optimality in path length.
No turn penalty was added, since the visibility graph already produces near-straight segments.

## Motion control

### Strategy

Controlling the robot require some kind of closed loop control, whether it's a P controller, an Astolfi or one of another kind. This loop could be implemented on the PC with Python, but this would induce a **huge latency** in the feedback loop, which would easily make the system **unstable**. That's why we prefered to implement the closed loop control law directly on the **Thymio** in **Aseba** to be aware of the features of the language. But this requires synchronization between the PC and the robot.

#### Initialization

A **Thymio** class handle this synchronization. It first loads the **Aseba** program and do a preprocessing step to load some constants into the program.

```python
class Thymio():

    def __init__(self, ...) -> None:
        
        # Init code
        ...

        # Calibration and program
        self.cal = calibration
        with open('thymio.aesl') as file:
            self.program = file.read().format(
                X0      = self.x,
                Y0      = self.y,
                X1      = int(np.round(x1)),
                Y1      = int(np.round(y1)),
                THETA0  = rad_to_lsb(self.theta),
                K_D     = self.cal.kD,
                K_THETA = self.cal.kTheta
            )
```

These **initialization constants** look like follow in the Thymio program.

```aseba
# Pose
var x_mm = {X0}
var x_um = 0
var y_mm = {Y0}
var y_um = 0
var theta = {THETA0}

# Reference
var r_x_mm = {X1}
var r_y_mm = {Y1}
```

Then a **context manager** handle connection and disconnection from the robot when the python code enter or exit a **with** statement :

```python
def __enter__(self) -> 'Thymio':

        # Initialize client
        self.client.__enter__()
        self.client.add_event_received_listener(self.on_event_received)
        
        # Connect to node
        self.node = aw(self.client.lock())
        ...
        
        # Compile program
        error = aw(self.node.compile(self.program))
        ...

        # Start program
        error = aw(self.node.run())
        ...
```

#### Communication

Communications from the **PC to the robot** are done by simply **writing variables** of the program. Below is an exemple for setting Thymios position and setpoint.

```python
def set_pose(self, x: float, y: float, theta: float) -> None:
    self.x = int(np.round(x))
    self.y = int(np.round(y))
    self.theta = theta
    aw(self.node.set_variables({'x_mm': [self.x], 'y_mm': [self.y], 'theta': [rad_to_lsb(theta)]}))

def set_target(self, x: float, y: float) -> None:
    self.blocked = False
    x = int(np.round(x))
    y = int(np.round(y))
    aw(self.node.set_variables({'state': [0], 'r_x_mm': [x], 'r_y_mm': [y]}))
```

Then communications from the **robot to the PC** is done by **emitting events** with data periodically. Below is an exemple to return odometry data.

```aseba
# Emit pose
emit pos [s, ms, x_mm, y_mm, theta, v, omega]
```

Then the **event is caught** in the thymio object :

```python
def on_event_received(self, node, event_name, event_data):

    match event_name:
        case 'pos':
            ...
        case 'blocked':
            ...
        case 'clear':
            ...
```

### Odometry

#### Pose

The Thymio robot in our use case has **3 DOF**, two translational and one rotational : $[x, y, \theta]^T$. The goal is to find the change in position over one time step :
$$
\begin{equation}
\begin{bmatrix}
    x_+ \\
    y_+ \\
    \theta_+
\end{bmatrix} =
\begin{bmatrix}
    x \\
    y \\
    \theta
\end{bmatrix} +
\begin{bmatrix}
    \Delta x \\
    \Delta y \\
    \Delta \theta
\end{bmatrix}
\end{equation}
$$

#### Change in heading

When moving one time step, the robot advance by $\Delta d_L$ and $\Delta d_R$ on each wheel. The track width $w$ being constant, we can consider that they are two arc length of two concentric circles of radiuses $w + r$ and $r$.

<p align="center">
    <img src="images/local-nav-odometry-heading.png" alt="odometry-heading" width="200"/>
</p>

With $\Delta d_L$ is the interior arc and $\Delta d_R$ the exterior arc, from the definition of the radian $\alpha = d / r$, we can solve the increment in angle that $\Delta d_L$ and $\Delta d_R$ describe :
$$
\begin{align}
\Delta \theta = \frac{\Delta d_L}{r} &= \frac{\Delta d_R}{w + r} \\
\Rightarrow \frac{r}{\Delta d_L} &= \frac{w + r}{\Delta d_R} \\
\Rightarrow r &= \frac{w \cdot \Delta d_L}{\Delta d_R - \Delta d_L} \\
\Rightarrow \Delta \theta &= \frac{\Delta d_R - \Delta d_L}{w}
\end{align}
$$

#### Change in position

To integrate the position, we could consider the same geometrical setup as before, but it would be a bit computationally heavy, and would lead to special cases (i.e. radius at infinity when moving in a straight line). The **mid-point rule** also known as **Runge-Kutta 2** (**RK2**, suggested by ChatGPT) is used instead.

<p align="center">
    <img src="images/local-nav-odometry-position.png" alt="odometry-position" width="150"/>
</p>

We consider that change of coordinate has followed a straight line $\Delta s$ at the mid-point heading $\theta_{mid}$.

$$
\begin{align}
\Delta s &= \frac{\Delta d_L + \Delta d_R}{2} \\
\theta_{mid} &= \theta + \frac{\Delta \theta}{2} \\
\Delta x &= \Delta s \cdot \cos \theta_{mid} \\
\Delta y &= \Delta s \cdot \sin \theta_{mid} \\
\end{align}
$$

#### Change in wheel distance

First, we solve the simple case of linear motion for each wheel. From the Thymio cheat sheet we get the constants :
$$
\begin{equation}
\begin{aligned}
\Delta t &= \frac{1}{100 \text{ Hz}} = \frac{10}{1000} \text{ s} \\
v_{mm/s} &= 20 \text{ cm/s} = 200 \text{ mm/s} \\
v_{lsb/s} &= 500 \text{ lsb/s}
\end{aligned}
\end{equation}
$$

To have a large enough full scale range of $\approx \pm 2^{15} = \pm 32'768 \approx \pm 32 \text{ m}$, the initial increment estimation is computed in $\text{mm}$ :
$$
\begin{align}
\Delta d_{mm} &= v_{lsb/s} \cdot \frac{v_{mm/s}}{v_{lsb/s}} \cdot \Delta t \\
&= v_{lsb/s} \cdot \frac{200}{500} \cdot \frac{10}{1000} \\
&= v_{lsb/s} \cdot \frac{2}{500} \\
\end{align}
$$
However, because of fixed point arithmetic, only the speeds $250$ and $500$ can be differentiated. They will respectively give increments of $\pm 1$ and $\pm 2$ on the position estimation, so almost $8$ bits of precision are lost. To take into acount those lost $8$ bits, we must estimate the increment in $\mu m$ :
$$
\begin{equation}
\Delta d_{\mu m} = 1000 \cdot \Delta d_{mm} = v_{lsb/s} \cdot \frac{20}{5} = v_{lsb/s} \cdot 4
\end{equation}
$$

After a bit of testing, it turns out that the given maximums speeds in $\text{mm/s}$ and $\text{lsb/s}$ are not exact. To find the actual constant needed to integrate distance, a setpoint was set $1 \text{ m}$ ahead of the robot. Then the constant was tuned until the robot moved the right distance. In the end, the distance integration constant was found to be :
$$
\begin{align}
\Delta d_{\mu m} = v_{lsb/s} \cdot 3.1254 \\
k_d = 3.1254 \approx \frac{31'254}{10'000}
\end{align}
$$

Which give a maximum speed closer to $15.6 \text{ cm/s}$.

In aseba, distance integration is then implemented like below :

```aseba
call math.muldiv(d_l_um, motor.left.speed, {K_D}, 10000)
call math.muldiv(d_r_um, motor.right.speed, {K_D}, 10000)
```

With **K_D** set to $31254$ and `math.muldiv` computing the multiplication in a 32 bit register to avoid a 16 bit overflow during the multiplication.

#### Change in robot heading

We computed earlier that we need to find the following angle increment :
$$
\begin{equation}
\Delta \theta = \frac{\Delta d_R - \Delta d_L}{w} \, rad
\end{equation}
$$

On the Thymio, angles use a fixed point representation in the range $[-\pi, \pi[$ mapped to $[-2^{15}, 2^{15}[$. So one radian is equal to :
$$
1 \, rad = \frac{2^{15}}{\pi} \approx 10'430
$$

If we try to deduce a constant with Thymio track width of $w = 95'000 \, \mu m$ to compute angles in these units, we get :
$$
\begin{align}
\Delta \theta &= \frac{\Delta d_R - \Delta d_L}{w} \, rad = (\Delta d_R - \Delta d_L) \cdot \frac{2^{15}}{\pi \cdot w} \\
k_\theta &= \frac{2^{15}}{\pi \cdot w} \approx \frac{10'430}{95'000} \approx \frac{1'098}{10'000}
\end{align}
$$

In aseba, this is the implemented like below :
```aseba
call math.muldiv(d_theta, d_r_um - d_l_um, {K_THETA}, 10000)
```

With **K_THETA** set to $1098$.

### Controller

#### Astolfi

The pose estimation is perfectly suited for an Astolfi controller, with $x_e$, $y_e$ and $\theta_e$ the pose error, and the change of coordiates :
$$
\begin{align}
\rho &= \sqrt{x_e^2 + y_e^2} \\
\alpha &= \text{atan2}(y_e, x_e) - \theta \\
\beta &= \theta_e - \alpha
\end{align}
$$

With $\rho$ is the distance to the target, $\alpha$ the target heading error, and $\beta$ the end heading error. The control law is :
$$
\begin{align}
v &= k_\rho \cdot \rho \\
\omega &= k_\alpha \cdot \alpha + k_\beta \cdot \beta \\
k_\rho &> 0, k_\beta < 0, k_\alpha - k_\rho > 0
\end{align}
$$

This controller works fine, but it tends to move in big arcs before getting to the target, which is not optimal in a tight environnement.

#### Tweaked astolfi

To fix this the controller must be changed a little. The end heading error $\beta$ is not needed, so it is removed. Then, to make clear that this is a different controller, $\rho$ is renamed as $e$ and $\alpha$ becomes $\varepsilon$ :
$$
\begin{align}
e &= \sqrt{x_e^2 + y_e^2} \\
\varepsilon &= \text{atan2}(y_e, x_e) - \theta
\end{align}
$$

Then, assuming $\varepsilon \in [-\pi, \pi[$, the controller is modified as :
$$
\begin{align}
v &= k_e \cdot e \cdot \left(\pi - |\varepsilon|\right) \\
\omega &= k_\varepsilon \cdot \varepsilon
\end{align}
$$

This **prevents the robot from moving** forward **until it's headed in the right direction**. Then it will act as a classic P controller. The speed PI controller will take care of the **soft start** while the position P controller does the **soft stop**. Implementation in Aseba looks like follow :

```aseba
# Compute carthesian error
e_x_mm = r_x_mm - x_mm
e_y_mm = r_y_mm - y_mm

# Compute polar error, clamp to avoid L2 norm overflow
call math.clamp(e_x_mm, e_x_mm, -127, 127)
call math.clamp(e_y_mm, e_y_mm, -127, 127)
call math.sqrt(e_mm, (e_x_mm * e_x_mm) + (e_y_mm * e_y_mm))
call math.atan2(epsilon, e_y_mm, e_x_mm)
epsilon -= theta

# Compute control outputs
call math.muldiv(v, e_mm, k_e_num, k_e_den)
call math.muldiv(v, v, 32767 - abs epsilon, 32767)
call math.muldiv(omega, epsilon, k_epsilon_num, k_epsilon_den)

# Clamp linear speed
call math.clamp(v, v, -200, 200)
left = v - omega
right = v + omega
```


## Local navigation

### Strategy

The **global navigation** is done **one waypoint at a time**.

<p align="center">
    <img src="images/local-nav-path-following.png" alt="path-following" width="200"/>
</p>

When detecting the obstacle with the horizontal proximity sensors, the avoidance trajectory should be pushed away from the obstacle, **tangeant** to it. This approach is simillar to a **potential field** repulsing the path.

<p align="center">
    <img src="images/local-nav-potential-field.png" alt="potential-field" width="400"/>
</p>

Knowing the path was generated with a **visibility graph**, we can assume each vertex is close to a **global obstacle**. The avoidance should hence be done on the **path exterior**, while considering that the robot is not perfectly on the path. It must intersect the trajectory to the global path to continue :

<p align="center">
    <img src="images/local-nav-avoidance-strategy.png" alt="avoidance-strategy" width="400"/>
</p>



### State machine

The **proximity sensors** being **far ahead of the rotation center** of the robot, with **small local obstacle**, directly implementing a potential field often lead to the robot to touching the obstacle while trying to avoid it. To prevent this, the robot should **probe** for the obstacle **tangeant**, then **nudge forward**. It should repeat these two steps until the nudge step intersect with the original path.

<p align="center">
    <img src="images/local-nav-avoidance-steps.png" alt="avoidance-steps" width="400"/>
</p>

This mechanism is implemented using a **state machine**. Two states are added to initially exit the path, in order to **avoid detecting a false intersection** with the path from the very start.

<p align="center">
    <img src="images/local-nav-avoidance-state-machine.png" alt="state-machine" width="300"/>
</p>

### Collision detection with the path

Knowing the **path the robot must follow** (blue), and the **next nudge step segment** (green), we can compute when and where the global path and the avoidance trajectory will **intersect** :

```python
def cross2d(a: np.ndarray, b: np.ndarray):
    return a[0] * b[1] - a[1] * b[0]

def trajectory_direction(path: np.ndarray) -> int:
    if path.shape[0] >= 3:
        return -1 if cross2d(path[1] - path[0], path[2] - path[1]) < 0 else 1
    else:
        return 1

def segments_intersection_point(s1: np.ndarray, s2: np.ndarray) -> np.ndarray | None:
    r = s1[1] - s1[0]
    s = s2[1] - s2[0]
    r_cross_s = cross2d(r, s)
    if abs(r_cross_s) < 1e-9:
        return None

    diff = s2[0] - s1[0]
    t = cross2d(diff, s) / r_cross_s
    u = cross2d(diff, r) / r_cross_s

    if 0 <= t <= 1 and 0 <= u <= 1:
        return s1[0] + t * r

    return None

def path_intersection_point(path: np.ndarray, segment: np.ndarray) -> tuple[np.ndarray | None, int | None]:
    for i in range(len(path) - 1):
        path_segment = path[i:(i + 2)]
        point = segments_intersection_point(segment, path_segment)
        if point is not None:
            return point, i
    return None, None
```

## Visualisation Dashboard

The project includes a dashboard that displays global planning, perception, and filtering outputs in a single view.  
It is built entirely with Matplotlib and serves as a monitoring interface. Its purpose is to visualise all system components clearly during experiments.

The dashboard consists of three panels:

- Left — Camera view

  Shows a live camera frame.  
  The global A* path and its nodes are projected into pixel space using the inverse homography.  
  Odometry (blue) and camera-based pose estimates (green) are drawn as circles with heading arrows and updated each time step.

- Middle — Global navigation map

  Displays the environment in world coordinates: original and inflated obstacles, start/goal markers, the A* path, and live odometry/camera trajectories.  
  This panel uses the output of `compute_global_path_with_debug(...)` together with the plotting utilities from `globalnav_plot.py`.

- Right — Filtering and performance

  The upper plot shows the EKF covariance diagonal $\sigma_x^2,\;\sigma_y^2,\;\sigma_\theta^2$ over time.  
  The lower plot shows the position error $\lVert \text{odom} - \text{EKF estimate} \rVert,$ which highlights drift and the corrective effect of camera updates.

Below the figure shows the visualisation board, including the three panels. (Note: Unfortunatly some axis labels are overlaid on other figures or slightly cut off)
<p align="center">
  <img src="images/Visualization_Board.jpeg" width="1000%">
</p>


## Integration

### Multithreading

To avoid blocking behaviour between the **robot**, the **camera** and the **dashboard**, we opted to use multithreading in our Python logic :
- **Main thread** : **render the dashboard** and **start** the two **other threads**
- **Thymio thread** : runs the **waypoint following** and **local navigation** logic, as well as the **Kalman filter**
- **Camera thread** : run the **camera acquisition** and robots **position detection**

The following flowchart exposes the overall logic :

<p align="center">
    <img src="images/overall-logic.png" alt="overall-logic" width="600"/>
</p>

### Constants tuning

In our demonstration, we used the following constants to tune Thymio's behavior :

```python
# Thymio calibration settings
GLOB_NAV_EPSILON = 90
WAYPOINT_POS_TOLERANCE = 20
NUDGE_LENGTH = 160
SOFT_KIDNAPPING_THRESHOLD = 15
HARD_KIDNAPPING_THRESHOLD = 200

# Kalman settings

TS = 0.1
Q = np.diag([0.04, 0.04, 7.5e-5])
R_cam = np.diag([0.05, 0.05, 1.2e-4])
P = np.diag([0.1, 0.1, 0.1])
```

These constants are all in millimeters. They were manually tuned while testing the robot:
- `GLOB_NAV_EPSILON` put a safety margin around global obtacle. Between a **half Thymios width** and **Thymios width** is best.
- `WAYPOINT_POS_TOLERANCE` is the distance in which we consider that the robot have reached the waypoint. **2 cm** is close enough to the waypoint and far enough so that the robot doesn't turn around the obstacle.
- `NUDGE_LENGTH` is the distance the robot move forward between each local navigation probing step. **One and a half Thymios length** gave the best result, not too close and not too far from the obstacle.
- `SOFT_KIDNAPPING_THRESHOLD` is the distance over which the robot internal position is corrected with the EKF estimate. **1.5 cm** is small enough to not deviate and large enough to avoid rollback.
- `HARD_KIDNAPPING_THRESHOLD` is the distance over which the global navigation will recompute a whole new path. **20 cm** is approximately the distance in which one Thymio and one global obstacle fits.

### Results

To validate our integration, we ran through several **scenarios** to assert wether all feature work together or not.

**Disclaimer**, these animations shows a nearly done version of the whole setup, but the Kalman filter is not finely tuned at this point, neither are the axes of the EKF covariance. But in the current code these issues are resolved.

Note: The speed of the GIFs is multiplied by a factor euqlas to five.

#### Normal run

The first scenario is a simple run where the robot goes from **start to finish**.

<p align="center">
    <img src="animations/normal-run.gif" alt="normal-run" width="680"/>
</p>

#### Local avoidance

Then we tested the local navigation, by putting **two obstacles** to see if the robot would avoid them.

<p align="center">
    <img src="animations/local-avoidance.gif" alt="local-avoidance" width="680"/>
</p>

#### Blind camera

We also tested blinding the camera position by **covering the aruco code on top of the Thymio**.

<p align="center">
    <img src="animations/camera-off.gif" alt="blind-camera" width="680"/>
</p>

#### Soft kidnapping

Then we tried **moving the robot a little off course** to see if position error is corrected by the sensor fusion of odometry and the camera.

<p align="center">
    <img src="animations/soft-kidnapping.gif" alt="soft-kidnapping" width="680"/>
</p>

#### Hard kidnapping

And finally, we tested the case where the robot is **moved completely off course**, in which case program is automatically restarted to recompute the global path.

<p align="center">
    <img src="animations/hard-kidnapping.gif" alt="hard-kidnapping" width="680"/>
</p>

## Conclusion
Despite the short time frame, we successfully implemented the complete system specified in the assignment and validated it across all five project scenarios. The project integrates global planning, perception, filtering, and control into a coherent pipeline capable of handling realistic disturbances and incomplete information.

#### Vision

The vision system was able to consistently detect the ArUco markers and global obstacles and convert them into a coordinate system compatible with the Thymio. Using ArUco markers for the board corners, robot, and goal allowed us to build a reliable pixel-to-millimeter homography and obtain accurate positions. The obstacle detection pipeline, combined with the morphological filtering and polygon simplification, produced stable obstacles for global planning. The main limitations were sensitivity to lighting and the poor camera feed quality, which caused jittery ArUco and obstacle detections. We minimized much of the instability using a stability buffer, though improved camera hardware would improve our project further.

#### Global Navigation

The visibility-graph + A* planner generated clean and safe paths from start to goal. Proper tuning of the inflation radius ε ensured that the robot avoided obstacles even when the controller deviated from the ideal geometric path. Further improvements could include turn-penalties or heading-aware costs to better model the robot’s dynamics.

#### Motion Control

With an adapted Astolfi controller, the robot followed the planned trajectory reliably, though not perfectly straight, an expected behaviour for nonholonomic robots. Wheel slip and controller overshoot required a slightly larger inflation margin to guarantee safe clearance. Additional tuning could further reduce path deviation.

#### Local Navigation

Our reactive obstacle-avoidance layer worked well for dynamic obstacles placed in the robot’s path, ensuring safe detours before rejoining the global plan. Some edge cases remain where the avoidance behaves less robustly and a more refined local strategy could further improve reliability.

#### Filtering

The EKF fused odometry and vision effectively. With camera updates available, drift was corrected consistently, when the camera was hidden, the estimate began to diverge as expected from wheel slip. Once vision returned, the filter quickly re-aligned with the true pose. This behaviour reflects the fundamental limitation of odometry-driven prediction: without external observations, drift is unavoidable, and no amount of tuning can remove this entirely.

#### Performance in the Five Scenarios

The system was validated across all required scenarios:

- A → B navigation: The robot followed the A* path and reached the goal reliably.
- Camera hidden: With vision unavailable, the EKF relied on odometry and drifted gradually, once vision returned, the estimate corrected immediately.
- Soft kidnapping: Small displacements were corrected quickly by the EKF.
- Hard kidnapping: Larger displacements caused temporary inconsistency, but the filter re-aligned once stable camera data was available.
- Local obstacle avoidance: Newly introduced obstacles were successfully avoided, and the robot rejoined the global path.

#### Final Remarks

The system behaved quiet robustly in all scenarios. The global and local planners, EKF fusion, and controller complemented each other well. With more time, future improvements could focus on turn-aware planning, richer local avoidance, and finer controller/filter tuning.

#### Implementation Note

A further improvement concerns the `demo.py` script. It currently acts as the central hub that ties together all modules and is functional but not optimised. A cleaner, more modular structure would make the system easier to maintain. However, given the limited time, this was a lower priority.

In [14]:
import matplotlib
import PyQt6
%matplotlib qt

In [15]:
from demo import main_thread

main_thread()

[['square' 'poly0' '3' 5.85490586135079e-14 2.341962344540316e-13]
 ['square' 'poly0' '2' 1255.0 0.0]
 ['square' 'poly0' '1' 1255.0 740.0]
 ['square' 'poly0' '0' 0.0 740.0]
 ['start' 'start_pt' 'Start' 1055.7626953125 131.2255401611328]
 ['goal' 'goal_pt' 'Goal' 130.6184844970703 427.2525939941406]
 ['vertex' 'poly1' '1' 913.1719360351562 12.926460266113281]
 ['vertex' 'poly1' '2' 929.9849853515625 20.24551010131836]
 ['vertex' 'poly1' '3' 933.7140502929688 102.10513305664062]
 ['vertex' 'poly1' '4' 894.6324462890625 145.0165252685547]
 ['vertex' 'poly1' '5' 886.0693969726562 137.29537963867188]
 ['vertex' 'poly1' '6' 867.781005859375 97.38079071044922]
 ['vertex' 'poly1' '7' 897.1227416992188 22.013795852661133]
 ['vertex' 'poly2' '1' 873.9403076171875 227.5059814453125]
 ['vertex' 'poly2' '2' 909.9732055664062 298.6995849609375]
 ['vertex' 'poly2' '3' 886.6696166992188 324.10345458984375]
 ['vertex' 'poly2' '4' 870.3954467773438 324.89556884765625]
 ['vertex' 'poly2' '5' 844.83471679